# GPT-2 MiniKV KV Cache Compression PoC
GPU'da GPT-2 üzerinde layer-discriminative KV cache quantization testi.


In [ ]:
import json, math, time
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache


In [ ]:
TEXTS = [
    "The transformer architecture revolutionized natural language processing by introducing "
    "self-attention mechanisms that capture long-range dependencies in text. These models "
    "process sequences in parallel making training much more efficient than recurrent "
    "neural networks. Later advances like GPT BERT and T5 showed that scaling up these "
    "architectures leads to remarkable improvements across many language tasks. The key "
    "innovation was the self-attention mechanism which computes weighted representations "
    "of all positions in the input sequence allowing the model to understand context.",
    "Large language models face a critical bottleneck during text generation: the key-value "
    "cache grows linearly with sequence length. For billion-parameter models serving "
    "thousands of concurrent users this memory cost becomes substantial. Techniques like "
    "quantization pruning and sparse attention have been developed to address this challenge. "
    "The most promising approaches include KV cache quantization which reduces memory "
    "footprint with minimal quality loss and attention pruning which removes redundant heads.",
    "Knowledge distillation is a technique where a smaller student model learns to mimic "
    "the behavior of a larger teacher model. This allows deploying lightweight models that "
    "retain much of the performance of their larger counterparts. Recent work has shown "
    "that combining distillation with quantization can produce extremely efficient models "
    "suitable for edge devices and real-time applications.",
]


In [ ]:
class Quantizer:
    def __init__(self, n_layers, bc):
        self.n = n_layers
        self.bc = bc if len(bc) == n_layers else bc + [4] * (n_layers - len(bc))
        self.state = [None] * n_layers

    def quantize(self, t, li):
        b = self.bc[li]
        if b >= 16:
            self.state[li] = {"bits": 16}
            return t, 1.0
        mn = t.min(-1, True).values
        mx = t.max(-1, True).values
        s = (mx - mn).clamp(1e-8) / (2**b - 1)
        zp = mn
        q = ((t - zp) / s).round().clamp(0, 2**b - 1).to(torch.uint8)
        self.state[li] = {"s": s, "zp": zp, "bits": b}
        return q, q.numel() / (t.numel() * 2)

    def dequant(self, q, li):
        st = self.state[li]
        return q if st["bits"] >= 16 else q.float() * st["s"] + st["zp"]

    def apply(self, cache: DynamicCache):
        qc = DynamicCache()
        for li in range(self.n):
            k = cache.key_cache[li]; v = cache.value_cache[li]
            qk, _ = self.quantize(k, li)
            qv, _ = self.quantize(v, li)
            dk = self.dequant(qk, li).to(k.dtype)
            dv = self.dequant(qv, li).to(v.dtype)
            qc.update(dk, dv, k.size(2))
        return qc

    def ratio(self, cache: DynamicCache):
        orig = sum(cache.key_cache[li].numel() * cache.key_cache[li].element_size() +
                   cache.value_cache[li].numel() * cache.value_cache[li].element_size()
                   for li in range(self.n))
        q_total = 0
        for li in range(self.n):
            b = self.bc[li]
            if b >= 16:
                k, v = cache.key_cache[li], cache.value_cache[li]
                q_total += k.numel()*k.element_size() + v.numel()*v.element_size()
            else:
                q_total += cache.key_cache[li].numel() + cache.value_cache[li].numel()
        return orig / q_total, orig / (1024**2), q_total / (1024**2)


In [ ]:
@torch.no_grad()
def main():




In [ ]:
# ── Push results to GitHub repo ──
import requests, base64, json

# 👇 Token eklenecek yer — aşağıdaki tırnakların arasına token yapıştır, sonra Run
GITHUB_TOKEN = "YOUR_TOKEN_HERE"

if GITHUB_TOKEN != "YOUR_TOKEN_HERE":
    results_path = "pocs/gpt2-minikv/gpt2_results_kaggle.json"
    url = f"https://api.github.com/repos/ahmettas21/kvforge/contents/{results_path}"
    content = json.dumps(results, indent=2)
    data = {
        "message": "GPT-2 MiniKV PoC results (Kaggle GPU)",
        "content": base64.b64encode(content.encode()).decode(),
    }
    # Get existing file SHA
    r_get = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if r_get.status_code == 200:
        data["sha"] = r_get.json()["sha"]
    r = requests.put(url, json=data,
                     headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if r.status_code in (200, 201):
        print(f"\n  📤 Results → {results_path}")
        print(f"     Commit: {r.json()['commit']['html_url']}")
    else:
        print(f"\n  ⚠️  Push failed: {r.status_code}")
        print(f"     {r.text[:300]}")
else:
    print("\n  ⏭️  Push skipped (no token)")

print("\n" + "=" * 80)
print("  ✅ Done!")
print("=" * 80)
